# PART IV: Clustering

In [ ]:
import sklearn as sk
import pandas as pd

Clustering Techniques: Use the scikit learn clustering library: https://scikit-learn.org/stable/modules/clustering.htmlLinks to an external site. Note that this page contains very useful illustrations and overviews of the clustering methods, worth reading!
K-means clustering: cluster.KMeansLinks to an external site..
Hierarchical clustering: Experiment with "ward", "complete", "average", and "single" linkage: cluster.AgglomerativeClusteringLinks to an external site..
DBSCAN clustering: Run experiments to find good parameter values. cluster.DBSCANLinks to an external site..

## Importing data

In [ ]:
df = pd.read_parquet("basic_processed_data.parquet")

## Data Pre-processing

Scale all continuous attributes to a 0 to 1 scale using MinMaxScaler

In [ ]:
from sklearn.preprocessing import MinMaxScaler

numeric_columns = df.select_dtypes(include="number").columns
df[numeric_columns] = MinMaxScaler().fit_transform(df[numeric_columns])

Take 10% subset

In [ ]:
from sklearn.model_selection import train_test_split

sample_df, _ = train_test_split(
    df,
    train_size=0.1,
    stratify=df["Severity"],
    random_state=42,
)

X = sample_df.drop(columns=["Severity"])
y = sample_df["Severity"]

Drop boolean values

In [ ]:
boolean_columns = X.select_dtypes(include="boolean").columns
X.drop(columns = boolean_columns, inplace = True)

## Clustering

In [ ]:
from sklearn.base import clone
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.feature_selection import f_classif
import matplotlib.pyplot as plt
import numpy as np

def make_cluster_plot(model, X, ax=None, title=None):
    standalone = ax is None
    scores = np.zeros(X.shape[1])
    columns = X.columns[X.var(axis=0) > 0]
    if len(columns) and np.unique(model.labels_).size > 1:
        with np.errstate(divide="ignore", invalid="ignore"):
            scores[X.columns.get_indexer(columns)] = f_classif(
                X[columns], model.labels_
            )[0]
    scores = np.nan_to_num(scores, nan=0.0, posinf=1e6, neginf=0.0)
    positions = np.argsort(scores)[-10:]

    if standalone:
        fig, ax = plt.subplots(figsize=(9, max(4, 0.45 * len(positions))))
    ax.barh(X.columns[positions], scores[positions], color="teal")
    ax.set(
        xlabel="ANOVA F-score (higher = stronger cluster separation)",
        ylabel="Feature",
        title=title or "Features most associated with cluster assignments",
    )
    if standalone:
        fig.tight_layout()
        return plt
    return ax


def make_cluster_plot_panel(models, X, labels=None):
    models = list(models)

    columns = min(2, len(models))
    rows = (len(models) + columns - 1) // columns
    fig, axes = plt.subplots(rows, columns, figsize=(9 * columns, 4.5 * rows), squeeze=False)
    for model, label, ax in zip(models, labels, axes.flat):
        make_cluster_plot(model, X, ax, label)
    for ax in axes.flat[len(models):]:
        ax.remove()
    fig.suptitle("Feature separation across clustering models")
    fig.tight_layout()
    return fig, axes


def compare_clustering_parameter(X, estimator, parameter, values, **fixed_params):
    base_model = estimator(**fixed_params)
    values = list(values)

    models = [
        clone(base_model).set_params(**{parameter: value}).fit(X)
        for value in values
    ]
    labels = [f"{type(base_model).__name__}: {parameter}={value}" for value in values]
    return make_cluster_plot_panel(models, X, labels)

### K-means Clustering

In [ ]:
cluster_plot_fig, cluster_plot_axes = compare_clustering_parameter(
    X,
    KMeans,
    parameter="n_clusters",
    values=[2, 10, 50, 200],
    random_state = 0
)
plt.show()

In [ ]:
cluster_plot_fig, cluster_plot_axes = compare_clustering_parameter(
    X,
    AgglomerativeClustering,
    parameter="linkage",
    values=['ward', 'average', 'complete', 'single']
)
plt.show()